# When debate pays, and when it costs 3x for nothing [Step 06.04]

> **MLCourse - Agentic AI - Agent Patterns**

Three methods, four graded tasks, one deterministic grader, identical instrumentation:

1. **single** - one call. The baseline.
2. **self-consistency (n=3)** - three independent samples at temperature 0.9,
   majority vote. Three calls, no agent sees another.
3. **debate** - two personas, one rebuttal round each, then a judge. Five calls,
   and the round-2 prompts carry the whole transcript.

Whatever the numbers say, we report them. A module that can only demonstrate its
technique winning is not teaching you how to evaluate one.

### Key takeaways

- Cost is **known in advance** (calls x tokens). Benefit is **not**, and must be
  measured per task family.
- The interesting statistic is not mean accuracy, it is the **per-task
  win/tie/loss** breakdown against the baseline.
- Four tasks cannot separate methods that differ by less than about one task. Say so.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
# Debate makes 5 calls per task. Four tasks x three methods is ~36 calls, so we pace
# generously to stay under the 8000 TPM free tier while other work may be running.
PACE = 4.0
print("PACE =", PACE, "seconds")

PACE = 4.0 seconds


### The task set


In [ ]:
# Four questions with a single, checkable numeric answer. They are deliberately of
# the "first instinct is wrong" family: the point of debate is supposed to be that
# a second agent catches what the first one missed.
#
# A DETERMINISTIC grader matters more than the questions. If an LLM grades, you are
# measuring the grader as much as the method.

import re

TASKS = [
    dict(id="bat_ball",
         q="A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
           "How much does the ball cost, in dollars?",
         answer=0.05),
    dict(id="widgets",
         q="If 5 machines take 5 minutes to make 5 widgets, how many minutes would "
           "100 machines take to make 100 widgets?",
         answer=5),
    dict(id="strawberry",
         q="How many times does the letter 'r' appear in the word 'strawberry'?",
         answer=3),
    dict(id="avg_speed",
         q="A car drives 60 km at 30 km/h, then another 60 km at 60 km/h. "
           "What is its average speed over the whole trip, in km/h?",
         answer=40),
]

NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")


def grade(text, expected, tol=1e-6):
    """Deterministic grader: take the LAST number in the reply and compare.

    Every method in this module is told to end with 'FINAL: <number>', so the last
    number is the stated answer. No LLM opinion is involved anywhere in scoring.
    """
    nums = NUM_RE.findall((text or "").replace(",", "").replace("$", ""))
    if not nums:
        return False, None
    got = float(nums[-1])
    return abs(got - expected) <= max(tol, abs(expected) * 1e-6), got


ANSWER_RULE = ("Think briefly, then end your reply with a line of exactly the form "
               "'FINAL: <number>' and nothing after it.")

print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-11s expected %s" % (t["id"], t["answer"]))


### Instrumentation


In [ ]:
# Every method is wrapped in one of these so the comparison in notebook 04 is
# apples-to-apples: same tasks, same grader, same token meter.

class Meter:
    def __init__(self, name):
        self.name = name
        self.calls = 0
        self.tin = 0
        self.tout = 0
        self.seconds = 0.0
        self._sleeps = 0.0

    def record(self, msg):
        u = msg.usage_metadata or {}
        self.calls += 1
        self.tin += u.get("input_tokens", 0)
        self.tout += u.get("output_tokens", 0)
        self._sleeps += PACE
        return msg

    @property
    def tokens(self):
        return self.tin + self.tout

    @property
    def cost(self):
        return usd(self.tin, self.tout)


def timed(meter, fn, *a, **kw):
    t0 = time.time()
    out = fn(*a, **kw)
    meter.seconds += time.time() - t0
    return out


### 1. The three methods

Each returns `(answer_number, list_of_messages)` so the meter can charge it
correctly.

In [5]:
from collections import Counter

single_llm = make_llm(temperature=0.0, max_tokens=280)
sample_llm = make_llm(temperature=0.9, max_tokens=280)
deb_a_llm = make_llm(temperature=0.0, max_tokens=260)
deb_b_llm = make_llm(temperature=0.0, max_tokens=260)
judge_llm = make_llm(temperature=0.0, max_tokens=200)

PERSONA_A = ("You are Agent A, a fast and confident problem solver. At most three "
             "short sentences, then the final line.")
PERSONA_B = ("You are Agent B, a careful sceptic. Assume the obvious answer to this "
             "kind of question is a trap and verify it explicitly. At most three "
             "short sentences, then the final line.")
JUDGE_SYSTEM = ("You are an impartial judge. Decide which position is arithmetically "
                "and logically correct. Ignore confidence, length and style. "
                "One sentence, then 'VERDICT: A' or 'VERDICT: B', then 'FINAL: <number>'.")


def m_single(q):
    m = safe_invoke(single_llm, [("user", q + " " + ANSWER_RULE)])
    _, n = grade(m.content, 0)
    return n, [m]


def m_self_consistency(q, n=3):
    msgs = []
    for _ in range(n):
        msgs.append(safe_invoke(sample_llm, [("user", q + " " + ANSWER_RULE)]))
    nums = [grade(m.content, 0)[1] for m in msgs]
    c = Counter(x for x in nums if x is not None)
    return (c.most_common(1)[0][0] if c else None), msgs


def m_debate(q):
    a1 = safe_invoke(deb_a_llm, [("system", PERSONA_A), ("user", q + " " + ANSWER_RULE)])
    b1 = safe_invoke(deb_b_llm, [("system", PERSONA_B), ("user", q + " " + ANSWER_RULE)])

    def rebut(llm, persona, own, other, name):
        p = ("%s\n\nYour own earlier answer:\n%s\n\n%s argued:\n%s\n\n"
             "If %s is right, change your answer. If you are right, name the exact "
             "step of their reasoning that is wrong. %s"
             % (q, own, name, other, name, ANSWER_RULE))
        return safe_invoke(llm, [("system", persona), ("user", p)])

    a2 = rebut(deb_a_llm, PERSONA_A, a1.content, b1.content, "Agent B")
    b2 = rebut(deb_b_llm, PERSONA_B, b1.content, a1.content, "Agent A")

    jm = safe_invoke(judge_llm, [("system", JUDGE_SYSTEM),
                                 ("user", "QUESTION:\n%s\n\nPOSITION A:\n%s\n\n"
                                          "POSITION B:\n%s" % (q, a2.content, b2.content))])
    _, n = grade(jm.content, 0)
    # If the judge failed to state a number, fall back to the winning position's
    # number - a parsing detail, not a second opinion.
    if n is None:
        v = re.search(r"VERDICT:\s*([AB])", jm.content)
        src = a2 if (v and v.group(1) == "A") else b2
        _, n = grade(src.content, 0)
    return n, [a1, b1, a2, b2, jm]


METHODS = [("single", m_single), ("self_consistency@3", m_self_consistency),
           ("debate", m_debate)]
print("methods:", [m[0] for m in METHODS])

methods: ['single', 'self_consistency@3', 'debate']


### 2. The run

This is the slow cell - roughly 36 paced calls. Progress prints per task so you can
watch it.

In [6]:
results = {name: {"meter": Meter(name), "per_task": {}} for name, _ in METHODS}

for t in TASKS:
    print("-" * 70)
    print("TASK", t["id"], "| truth", t["answer"])
    for name, fn in METHODS:
        meter = results[name]["meter"]
        t0 = time.time()
        num, msgs = fn(t["q"])
        meter.seconds += time.time() - t0 - PACE * len(msgs)
        for m in msgs:
            meter.record(m)
        ok = num is not None and abs(num - t["answer"]) <= max(1e-6, abs(t["answer"]) * 1e-6)
        results[name]["per_task"][t["id"]] = (ok, num)
        print("  %-20s -> %-8s %s" % (name, num, "OK" if ok else "wrong"))
print("-" * 70)
print("done")

----------------------------------------------------------------------
TASK bat_ball | truth 0.05


  single               -> 0.05     OK


  self_consistency@3   -> 0.05     OK


  debate               -> 0.05     OK
----------------------------------------------------------------------
TASK widgets | truth 5


  single               -> 5.0      OK


...

  self_consistency@3   -> 3.0      OK


  debate               -> 3.0      OK
----------------------------------------------------------------------
TASK avg_speed | truth 40


  single               -> 2.0      wrong


  self_consistency@3   -> 3.0      wrong


  debate               -> 40.0     OK
----------------------------------------------------------------------
done


### 3. The result table


In [7]:
n = len(TASKS)
base = results["single"]

print("=" * 78)
print("%-20s %8s %7s %8s %10s %11s %10s"
      % ("method", "correct", "acc", "calls", "tokens", "cost USD", "sec"))
print("-" * 78)
for name, _ in METHODS:
    r = results[name]
    m = r["meter"]
    c = sum(1 for ok, _ in r["per_task"].values() if ok)
    print("%-20s %5d/%-2d %7.3f %8d %10d %11.6f %10.1f"
          % (name, c, n, c / n, m.calls, m.tokens, m.cost, m.seconds))
print("=" * 78)

method                correct     acc    calls     tokens    cost USD        sec
------------------------------------------------------------------------------
single                   3/4    0.750        4       1163    0.000603        2.5
self_consistency@3       3/4    0.750       12       3511    0.001823        7.3
debate                   4/4    1.000       20       4849    0.001696        6.7


In [8]:
bm = base["meter"]
bc = sum(1 for ok, _ in base["per_task"].values() if ok)

print("relative to the single-pass baseline")
print("-" * 70)
print("%-20s %12s %12s %14s" % ("method", "acc delta", "token mult", "win/tie/loss"))
print("-" * 70)
for name, _ in METHODS:
    if name == "single":
        continue
    r = results[name]
    c = sum(1 for ok, _ in r["per_task"].values() if ok)
    win = tie = loss = 0
    for t in TASKS:
        b_ok = base["per_task"][t["id"]][0]
        m_ok = r["per_task"][t["id"]][0]
        if m_ok and not b_ok:
            win += 1
        elif b_ok and not m_ok:
            loss += 1
        else:
            tie += 1
    print("%-20s %+11.3f %11.2fx %14s"
          % (name, (c - bc) / n, r["meter"].tokens / bm.tokens,
             "%d / %d / %d" % (win, tie, loss)))
print("-" * 70)

relative to the single-pass baseline
----------------------------------------------------------------------
method                  acc delta   token mult   win/tie/loss
----------------------------------------------------------------------
self_consistency@3        +0.000        3.02x      0 / 4 / 0
debate                    +0.250        4.17x      1 / 3 / 0
----------------------------------------------------------------------


In [9]:
print("per-task detail (this is what you should actually read)")
print("-" * 74)
print("%-12s %8s %14s %20s %14s" % ("task", "truth", "single", "self_cons@3", "debate"))
print("-" * 74)
for t in TASKS:
    cells = []
    for name, _ in METHODS:
        ok, num = results[name]["per_task"][t["id"]]
        cells.append("%s %s" % (num, "OK" if ok else "X"))
    print("%-12s %8s %14s %20s %14s" % (t["id"], t["answer"], cells[0], cells[1], cells[2]))
print("-" * 74)

per-task detail (this is what you should actually read)
--------------------------------------------------------------------------
task            truth         single          self_cons@3         debate
--------------------------------------------------------------------------
bat_ball         0.05        0.05 OK              0.05 OK        0.05 OK
widgets             5         5.0 OK               5.0 OK         5.0 OK
strawberry          3         3.0 OK               3.0 OK         3.0 OK
avg_speed          40          2.0 X                3.0 X        40.0 OK
--------------------------------------------------------------------------


### 4. Reading the result

Work through the table above rather than skipping to a conclusion. The three
outcomes you might be looking at, and what each means:

**(a) Every method scores the same.** The most common outcome on well-known puzzle
questions, because a modern instruction-tuned model has seen them and answers them
correctly on the first pass. Debate then bought nothing and cost several times the
tokens. *This is a real and useful finding* - it says your task family does not have
the failure mode debate fixes. Do not deploy it.

**(b) Debate wins on one or two tasks and loses on none.** Encouraging, and still
not significant at n=4. The right next step is to run the same harness on 100+ real
tasks, not to ship.

**(c) Debate loses somewhere.** Look at that task's transcript. Almost always it is
capitulation (notebook 02): an agent that was right folded when contradicted, and
the judge ratified the fold. This is the characteristic way debate makes things
worse.

### The cost side is not in doubt

Whatever accuracy did, the token multiplier above is real and reproducible. Note it
is **larger than the call-count ratio**, because rebuttal and judge prompts carry
the transcript. Budget for the tokens, not the calls.

### When debate is genuinely worth it

- The answer is **checkable but only expensively** (a human review, a slow test
  suite) - debate is cheaper than the check and filters what reaches it.
- The task has a **known trap** that a sceptic persona is primed to catch.
- The **agents differ substantively** - different models, different tools, different
  retrieved context. Two personas of one model at temperature 0 is the weakest form
  of debate there is, and it is what we ran here.
- The cost of a wrong answer is much larger than 5x an inference call.

### When it is not

- High-volume, low-stakes traffic. Use `../05_semantic_routing` instead and spend
  the savings elsewhere.
- Tasks you cannot grade. You will never know if it helped.
- Any case where a **deterministic verifier** exists - run the verifier. A unit
  test, a schema check or a calculator beats any number of arguing language models,
  and this is the lesson `../../02_langgraph/08_advanced_reasoning_patterns/02_reflexion`
  makes as well.

### Where to go next

- [`../07_llm_as_judge`](../07_llm_as_judge) - the judge you just trusted, taken
  apart: rubrics, position bias, self-preference, and calibration against humans.
- [`../08_agent_benchmarks`](../08_agent_benchmarks) - how to run this kind of
  comparison properly, with pass@k and run-to-run variance.
- [`../../02_langgraph/06_multi_agent_systems`](../../02_langgraph/06_multi_agent_systems) -
  the supervisor and swarm patterns this module contrasts with.